## Name: Keshab Bhattarai StudentID: 240453

## Import PySpark Libraries and Create PySpark Session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

In [2]:
spark = SparkSession.builder.appName("Data Analysis Student").master("local[*]").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/25 09:13:34 WARN Utils: Your hostname, Keshabs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.1.25.135 instead (on interface en0)
26/05/25 09:13:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 09:13:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/25 09:13:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Loading and Inspection

In [4]:
df_infer = spark.read.csv("students_data.csv", header=True, inferSchema=True)
df_infer.printSchema()
df_infer.show(5)

root
 |-- student_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- gpa: double (nullable = true)
 |-- scholarship: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)

+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|student_id|         name|gender|age|      department|year| gpa|scholarship|     city|               email|
+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|       101| Aarav Sharma|  Male| 20|Computer Science|   2| 3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       102|  Priya Thapa|Female| 22|        Business|   4| 3.2|         No|  Pokhara| priya.thapa@uni.edu|
|       103|  Rohan Karki|  Male| 21|     Engineering|   3|NULL|       

In [5]:
print("Total Rows:", df_infer.count())
print("Total Columns:", len(df_infer.columns))

Total Rows: 25
Total Columns: 10


In [6]:
schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("gpa", FloatType(), True),
    StructField("scholarship", StringType(), True),
    StructField("city", StringType(), True),
    StructField("email", StringType(), True)
])

df = spark.read.csv("students_data.csv", header=True, schema=schema)
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- gpa: float (nullable = true)
 |-- scholarship: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)



## Null Value Handling

In [9]:
null_counts = df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])
null_counts.show()

+----------+----+------+---+----------+----+---+-----------+----+-----+
|student_id|name|gender|age|department|year|gpa|scholarship|city|email|
+----------+----+------+---+----------+----+---+-----------+----+-----+
|         0|   0|     0|  2|         1|   0|  3|          0|   0|    0|
+----------+----+------+---+----------+----+---+-----------+----+-----+



In [11]:
from pyspark.sql.functions import col, when
df.filter(col("gpa").isNull()).show()

+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+
|student_id|          name|gender|age| department|year| gpa|scholarship|    city|               email|
+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+
|       103|   Rohan Karki|  Male| 21|Engineering|   3|NULL|         No|Lalitpur| rohan.karki@uni.edu|
|       110|  Nisha Tamang|Female| 20|       Arts|   2|NULL|         No|Lalitpur|nisha.tamang@uni.edu|
|       122|Dipika Niroula|Female| 20|   Business|   2|NULL|         No|  Dharan|dipika.niroula@un...|
+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+



In [12]:
# Calculate the average GPA of all non-null students
mean_gpa_val = df.select(F.avg("gpa")).collect()[0][0]
print("Calculated Mean GPA:", mean_gpa_val)

# Fill missing data
df_filled = df.fillna({
    "gpa": mean_gpa_val,
    "department": "Undeclared",
    "age": 20
})
df_filled.show(5)

Calculated Mean GPA: 3.304545456712896
+----------+-------------+------+---+----------------+----+---------+-----------+---------+--------------------+
|student_id|         name|gender|age|      department|year|      gpa|scholarship|     city|               email|
+----------+-------------+------+---+----------------+----+---------+-----------+---------+--------------------+
|       101| Aarav Sharma|  Male| 20|Computer Science|   2|      3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       102|  Priya Thapa|Female| 22|        Business|   4|      3.2|         No|  Pokhara| priya.thapa@uni.edu|
|       103|  Rohan Karki|  Male| 21|     Engineering|   3|3.3045454|         No| Lalitpur| rohan.karki@uni.edu|
|       104|     Sita Rai|Female| 19|Computer Science|   1|      3.9|        Yes|Kathmandu|    sita.rai@uni.edu|
|       105|Bikash Gurung|  Male| 23|        Business|   4|      2.7|         No|   Butwal|bikash.gurung@uni...|
+----------+-------------+------+---+----------------+---

## Selecting and Filtering

In [13]:
df_filled.select("name", "department", "gpa").show()

+----------------+----------------+---------+
|            name|      department|      gpa|
+----------------+----------------+---------+
|    Aarav Sharma|Computer Science|      3.8|
|     Priya Thapa|        Business|      3.2|
|     Rohan Karki|     Engineering|3.3045454|
|        Sita Rai|Computer Science|      3.9|
|   Bikash Gurung|        Business|      2.7|
|  Anita Shrestha|     Engineering|      3.5|
|   Deepak Pandey|            Arts|      3.1|
|     Kavya Joshi|Computer Science|      3.7|
|    Suresh Limbu|     Engineering|      2.9|
|    Nisha Tamang|            Arts|3.3045454|
|   Prakash Bista|        Business|      3.4|
|     Manisha Oli|Computer Science|      3.6|
| Rajesh Adhikari|     Engineering|      3.0|
|    Sunita Magar|            Arts|      2.8|
|    Kiran Poudel|Computer Science|      3.3|
|     Rupa Basnet|      Undeclared|      3.7|
|  Anil Chaudhary|     Engineering|      2.6|
|    Puja Koirala|        Business|      3.5|
|     Nabin Dahal|            Arts

In [14]:
df_filled.filter((col("department") == "Computer Science") & (col("gpa") > 3.5)).show()

+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|student_id|            name|gender|age|      department|year|gpa|scholarship|     city|               email|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|       101|    Aarav Sharma|  Male| 20|Computer Science|   2|3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       104|        Sita Rai|Female| 19|Computer Science|   1|3.9|        Yes|Kathmandu|    sita.rai@uni.edu|
|       108|     Kavya Joshi|Female| 20|Computer Science|   2|3.7|        Yes|  Pokhara| kavya.joshi@uni.edu|
|       112|     Manisha Oli|Female| 21|Computer Science|   3|3.6|        Yes|  Pokhara| manisha.oli@uni.edu|
|       120|Kabita Bhattarai|Female| 22|Computer Science|   4|3.9|        Yes|Bhaktapur|kabita.bhattarai@...|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+



In [15]:
df_filled.filter((col("year").isin([3, 4])) & (col("scholarship") == "Yes")).show()

+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|student_id|            name|gender|age|      department|year|gpa|scholarship|     city|               email|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|       111|   Prakash Bista|  Male| 24|        Business|   4|3.4|        Yes|Kathmandu|prakash.bista@uni...|
|       112|     Manisha Oli|Female| 21|Computer Science|   3|3.6|        Yes|  Pokhara| manisha.oli@uni.edu|
|       116|     Rupa Basnet|Female| 23|      Undeclared|   4|3.7|        Yes|  Pokhara| rupa.basnet@uni.edu|
|       120|Kabita Bhattarai|Female| 22|Computer Science|   4|3.9|        Yes|Bhaktapur|kabita.bhattarai@...|
|       121|      Suman Giri|  Male| 21|     Engineering|   3|3.4|        Yes|Kathmandu|  suman.giri@uni.edu|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+



In [16]:
df_graded = df_filled.withColumn(
    "grade",
    when(col("gpa") >= 3.7, "Distinction")
    .when(col("gpa") >= 3.0, "Merit")
    .otherwise("Pass")
)
df_graded.show(5)

+----------+-------------+------+---+----------------+----+---------+-----------+---------+--------------------+-----------+
|student_id|         name|gender|age|      department|year|      gpa|scholarship|     city|               email|      grade|
+----------+-------------+------+---+----------------+----+---------+-----------+---------+--------------------+-----------+
|       101| Aarav Sharma|  Male| 20|Computer Science|   2|      3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|Distinction|
|       102|  Priya Thapa|Female| 22|        Business|   4|      3.2|         No|  Pokhara| priya.thapa@uni.edu|      Merit|
|       103|  Rohan Karki|  Male| 21|     Engineering|   3|3.3045454|         No| Lalitpur| rohan.karki@uni.edu|      Merit|
|       104|     Sita Rai|Female| 19|Computer Science|   1|      3.9|        Yes|Kathmandu|    sita.rai@uni.edu|Distinction|
|       105|Bikash Gurung|  Male| 23|        Business|   4|      2.7|         No|   Butwal|bikash.gurung@uni...|       Pass|


## Dropping and Renaming

In [17]:
df_dropped = df_graded.drop("email", "student_id")
print("Remaining Column Names:", df_dropped.columns)

Remaining Column Names: ['name', 'gender', 'age', 'department', 'year', 'gpa', 'scholarship', 'city', 'grade']


In [18]:
df_no_null_dept = df.dropna(subset=["department"])
df_unique = df_no_null_dept.dropDuplicates(["name", "department"])
df_unique.show(5)

+----------+--------------+------+---+----------------+----+---+-----------+---------+--------------------+
|student_id|          name|gender|age|      department|year|gpa|scholarship|     city|               email|
+----------+--------------+------+---+----------------+----+---+-----------+---------+--------------------+
|       101|  Aarav Sharma|  Male| 20|Computer Science|   2|3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       117|Anil Chaudhary|  Male| 21|     Engineering|   3|2.6|         No|Nepalgunj|anil.chaudhary@un...|
|       106|Anita Shrestha|Female| 20|     Engineering|   2|3.5|        Yes|Bhaktapur|anita.shrestha@un...|
|       105| Bikash Gurung|  Male| 23|        Business|   4|2.7|         No|   Butwal|bikash.gurung@uni...|
|       125|Bishal Ghimire|  Male| 23|     Engineering|   4|2.8|         No|Kathmandu|bishal.ghimire@un...|
+----------+--------------+------+---+----------------+----+---+-----------+---------+--------------------+
only showing top 5 rows


In [19]:
df_renamed = df_graded.withColumnRenamed("gpa", "grade_point_avg").withColumnRenamed("year", "study_year")
df_renamed.show(5)

+----------+-------------+------+---+----------------+----------+---------------+-----------+---------+--------------------+-----------+
|student_id|         name|gender|age|      department|study_year|grade_point_avg|scholarship|     city|               email|      grade|
+----------+-------------+------+---+----------------+----------+---------------+-----------+---------+--------------------+-----------+
|       101| Aarav Sharma|  Male| 20|Computer Science|         2|            3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|Distinction|
|       102|  Priya Thapa|Female| 22|        Business|         4|            3.2|         No|  Pokhara| priya.thapa@uni.edu|      Merit|
|       103|  Rohan Karki|  Male| 21|     Engineering|         3|      3.3045454|         No| Lalitpur| rohan.karki@uni.edu|      Merit|
|       104|     Sita Rai|Female| 19|Computer Science|         1|            3.9|        Yes|Kathmandu|    sita.rai@uni.edu|Distinction|
|       105|Bikash Gurung|  Male| 23|    

In [20]:
new_column_names = [c.replace("_", " ") for c in df_graded.columns]
df_spaced = df_graded.toDF(*new_column_names)
print("Spaced Column Names:", df_spaced.columns)
df_spaced.show(5)

Spaced Column Names: ['student id', 'name', 'gender', 'age', 'department', 'year', 'gpa', 'scholarship', 'city', 'email', 'grade']
+----------+-------------+------+---+----------------+----+---------+-----------+---------+--------------------+-----------+
|student id|         name|gender|age|      department|year|      gpa|scholarship|     city|               email|      grade|
+----------+-------------+------+---+----------------+----+---------+-----------+---------+--------------------+-----------+
|       101| Aarav Sharma|  Male| 20|Computer Science|   2|      3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|Distinction|
|       102|  Priya Thapa|Female| 22|        Business|   4|      3.2|         No|  Pokhara| priya.thapa@uni.edu|      Merit|
|       103|  Rohan Karki|  Male| 21|     Engineering|   3|3.3045454|         No| Lalitpur| rohan.karki@uni.edu|      Merit|
|       104|     Sita Rai|Female| 19|Computer Science|   1|      3.9|        Yes|Kathmandu|    sita.rai@uni.edu|Distinc

## Statistics and Aggregation

In [23]:
df_filled.describe("gpa", "age").show()

+-------+-------------------+------------------+
|summary|                gpa|               age|
+-------+-------------------+------------------+
|  count|                 25|                25|
|   mean|  3.304545450210571|              21.0|
| stddev|0.36909493967779344|1.3844373104863457|
|    min|                2.6|                19|
|    max|                3.9|                24|
+-------+-------------------+------------------+



In [24]:
df_filled.groupBy("department").agg(F.avg("gpa").alias("avg_gpa")).orderBy(col("avg_gpa").desc()).show()

+----------------+------------------+
|      department|           avg_gpa|
+----------------+------------------+
|      Undeclared| 3.700000047683716|
|Computer Science| 3.614285707473755|
|        Business|3.2209091186523438|
|            Arts|3.2009090423583983|
|     Engineering| 3.072077921458653|
+----------------+------------------+



In [25]:
df_filled.groupBy("gender", "scholarship").count().show()

+------+-----------+-----+
|gender|scholarship|count|
+------+-----------+-----+
|  Male|         No|   10|
|  Male|        Yes|    3|
|Female|         No|    5|
|Female|        Yes|    7|
+------+-----------+-----+



In [26]:
df_filled.groupBy("city").agg(
    F.count("student_id").alias("student_count"),
    F.avg("gpa").alias("avg_gpa"),
    F.sum(when(col("scholarship") == "Yes", 1).otherwise(0)).alias("scholarship_count")
).filter(col("student_count") > 2).show()

+---------+-------------+------------------+-----------------+
|     city|student_count|           avg_gpa|scholarship_count|
+---------+-------------+------------------+-----------------+
|  Pokhara|            5|3.5599999904632567|                4|
| Lalitpur|            4| 3.227272689342499|                0|
|Kathmandu|            8|3.3375000059604645|                4|
+---------+-------------+------------------+-----------------+



## Bonus Challenge

In [27]:
from pyspark.sql.window import Window

# Reload a fresh version of the base data
df_base = spark.read.csv("students_data.csv", header=True, schema=schema)

# Run complete sequence of operations in one pipeline block
pipeline_df = (
    df_base
    .dropna(how="all", subset=["gpa", "age"])
    .withColumn("gpa", F.coalesce(col("gpa"), F.avg("gpa").over(Window.partitionBy("department"))))
    .withColumn("grade", when(col("gpa") >= 3.7, "Distinction").when(col("gpa") >= 3.0, "Merit").otherwise("Pass"))
    .drop("email")
    .withColumnRenamed("gpa", "grade_point_avg")
    .orderBy(col("grade_point_avg").desc())
)

# Export the optimized final result to a Parquet directory
pipeline_df.write.parquet("output/students_clean.parquet", mode="overwrite")
print("Bonus production pipeline saved successfully as Parquet format!")

Bonus production pipeline saved successfully as Parquet format!


In [28]:
spark.stop()